In [0]:
# Configurações
catalog = "v_credit_medallion"
bronze_schema_name = "bronze"
silver_schema_name = "silver"

# Schemas completos
bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"

In [0]:
from pyspark.sql import functions as F

## Transformações — `tb_atendentes`

### Padronização de `nivel_atendimento`

- A coluna original apresenta cinco representações distintas para dois níveis de atendimento: `"1"`, `"Nível 1"`, `"2"`, `"Nível 2"` e `"N2"`.
- Todos os valores foram mapeados para inteiro (`1` ou `2`), eliminando variações textuais.

### Conversão de datas

- `data_admissao` e `data_desligamento` foram convertidas de string (`dd/MM/yyyy`) para o tipo `date`.
- Strings vazias em `data_desligamento` foram tratadas como `null`, representando atendentes sem desligamento registrado.

### Tipagem

- `id_atendente` foi convertido de string para inteiro.
- `status_atendente` foi mantido sem alteração, pois já apresenta valores consistentes (`ativo`/`inativo`).


In [0]:
# Leitura da tabela bruta na camada Bronze
df_atendentes = spark.table(f"{bronze_schema}.tb_atendentes")

df_atendentes_silver = (df_atendentes
    # Tipagem da chave primária para inteiro                    
    .withColumn("id_atendente", F.col("id_atendente").cast("int"))

    # Padronização das variações textuais de nível de atendimento para numérico (1 ou 2)
    .withColumn("nivel_atendimento",
        F.when(F.col("nivel_atendimento").isin("1", "Nível 1"), 1)
        .when(F.col("nivel_atendimento").isin("2", "Nível 2", "N2"), 2)
        .cast("int")
    )
    
    # Conversão de datas
    .withColumn("data_admissao", F.to_date("data_admissao", "dd/MM/yyyy"))
    .withColumn("data_desligamento",F.when(F.col("data_desligamento") == "", None)
    .otherwise(F.to_date("data_desligamento", "dd/MM/yyyy"))
    )
    
    # Padronização do status
    .withColumn("status_atendente", F.initcap(F.col("status_atendente")))

    .withColumn("_processed_at", F.current_timestamp())
    
    # Remoção do metadado de ingestão da camada Bronze
    .drop("_ingested_at")
)

(
    df_atendentes_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.tb_atendentes")
)

display(spark.table(f"{silver_schema}.tb_atendentes").limit(10))

%md
## Transformações — `tb_canais`

### Tipagem

- `id_canal` foi convertido de string para inteiro para servir como chave primária padronizada.
- As demais colunas (`nome_canal`, `tipo_canal` e `status_canal`) foram mantidas sem alteração por já apresentarem valores normalizados e consistentes.


In [0]:
# Leitura da tabela bruta na camada Bronze
df_canais = spark.table(f"{bronze_schema}.tb_canais")

df_canais_silver = (
    df_canais
    # Tipagem da chave primária
    .withColumn("id_canal", F.col("id_canal").cast("int"))
    
    # Padronização textual dos canais
    .withColumn("nome_canal",
        F.when(F.col("id_canal") == 1, "URA")
         .when(F.col("id_canal") == 2, "Chatbot")
         .when(F.col("id_canal") == 3, "Atendimento Inicial")
         .when(F.col("id_canal") == 4, "Atendimento Especializado")
         .otherwise(F.col("nome_canal"))
    )
    
    # Correção ortográfica do tipo de canal
    .withColumn("tipo_canal",
        F.when(F.col("tipo_canal") == "Autosservico", "Autosserviço")
         .otherwise(F.col("tipo_canal"))
    )
    
    # Padronização do status
    .withColumn("status_canal", F.initcap(F.col("status_canal")))
    
    .withColumn("_processed_at", F.current_timestamp())

    # Remoção do metadado de ingestão da camada Bronze
    .drop("_ingested_at")
)

# Gravação na Silver
(
    df_canais_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.tb_canais")
)

display(spark.table(f"{silver_schema}.tb_canais"))


## Transformações — `tb_pesquisa_satisfacao`

### Saneamento de `nota_atendimento`

- A escala de avaliação de satisfação do atendimento é de 1 a 5.
- Foram identificados valores textuais (`"NULL"` e vazios `""`), que representam pesquisas enviadas porém não respondidas pelo cliente.
- Valores fora do domínio válido (`-1`, `0` e `6`) foram identificados, decorrentes de falhas de canal (ex: digitação incorreta em URA/chatbot, campos de texto livre sem validação ou códigos de erro de sistema).
- **Tratamento:** todos os registros fora da faixa válida `[1, 5]` foram convertidos para `null`. As linhas foram preservadas para manter a integridade do histórico de pesquisas, garantindo que valores inválidos não distorçam cálculos de agregação e médias futuras.

### Tipagem

- `id_pesquisa` e `id_chamado` foram convertidos de string para inteiro.
- `nota_atendimento` foi tipada como inteiro.


In [0]:
# Leitura da tabela bruta na camada Bronze
df_pesquisa = spark.table(f"{bronze_schema}.tb_pesquisa_satisfacao")

df_pesquisa_silver = (df_pesquisa
    # Tipagem das chaves para inteiro
    .withColumn("id_pesquisa", F.col("id_pesquisa").cast("int"))
    .withColumn("id_chamado", F.col("id_chamado").cast("int"))
    
    # Mantém apenas notas no intervalo válido [1, 5]
    # Vazios, "NULL", -1, 0 e 6 tornam-se null automaticamente
    .withColumn("nota_atendimento",F.when(F.col("nota_atendimento").isin("1", "2", "3", "4", "5"), F.col("nota_atendimento").cast("int"))
    .otherwise(None)
    )
    
    .withColumn("_processed_at", F.current_timestamp())
    
    # Remoção do metadado de ingestão da camada Bronze
    .drop("_ingested_at")
)

(
    df_pesquisa_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.tb_pesquisa_satisfacao")
)

display(spark.table(f"{silver_schema}.tb_pesquisa_satisfacao").limit(10))

## Transformações — `tb_custos`

### Deduplicação

- Foram identificadas 781 linhas duplicadas exatas (mesmo `id_custo`, `id_chamado` e `custo`).
- As duplicatas foram removidas utilizando deduplicação no conjunto de dados.

### Padronização e saneamento de `custo`

- A coluna original apresentou múltiplos padrões de representação: prefixo monetário (`"R$ "`), sufixo textual (`"reais"`), espaços em branco e inconsistência no separador decimal (uso misto de vírgula e ponto).
- Foram removidos todos os caracteres não numéricos e padronizado o separador decimal para ponto.
- **Valores negativos:** foram identificados 249 registros com valores negativos. Como atendimentos operacionais geram custos positivos de telefonia e pessoal, assumiu-se que o sinal decorre de notação contábil de débito do sistema de origem. Aplicou-se a função de valor absoluto (`abs()`) para manter a despesa real sem subestimar o custo total da operação.
- A coluna foi convertida para tipo numérico decimal (`decimal(10,2)`).

### Tipagem

- `id_custo` e `id_chamado` foram convertidos de string para inteiro.


In [0]:
# Leitura da tabela bruta na camada Bronze
df_custos = spark.table(f"{bronze_schema}.tb_custos")

df_custos_silver = (df_custos
    # Remoção de duplicatas
    .dropDuplicates()
    
    # Tipagem das chaves para inteiro
    .withColumn("id_custo", F.col("id_custo").cast("int"))
    .withColumn("id_chamado", F.col("id_chamado").cast("int"))
    
    # Remove "R$", "reais" e espaços em branco
    .withColumn("custo_limpo", F.regexp_replace(F.col("custo"), r"[R\$\s]|reais", ""))

    # Substitui vírgula por ponto para permitir a conversão numérica
    .withColumn("custo_limpo",F.regexp_replace(F.col("custo_limpo"), ",", "."))

    # Converte para decimal e aplica abs() para corrigir sinais negativos
    .withColumn("custo",F.abs(F.col("custo_limpo").cast("decimal(10,2)")))
    .drop("custo_limpo")
    
    .withColumn("_processed_at", F.current_timestamp())
    
    # Remoção do metadado de ingestão da camada Bronze
    .drop("_ingested_at")
)

(
    df_custos_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.tb_custos")
)

display(spark.table(f"{silver_schema}.tb_custos").limit(10))

## Transformações — `tb_Chamados_Hora`

### 1. Nomenclatura e Tipagem
- Colunas renomeadas para *snake_case* minúsculo (`id_chamado`, `id_cliente`, `hora_abertura_chamado`, `hora_inicio_atendimento`, `hora_finalizacao_atendimento`).
- `id_chamado` tipado como `int`.
- `id_cliente` tipado como `bigint`.

### 2. Tratamento dos Horários
- Remoção do texto `" às "` entre data e hora e conversão para `timestamp` no padrão `dd/MM/yyyy HH:mm:ss`.

### 3. Decisão sobre Nulos em `hora_inicio_atendimento`
- Conforme comprovado na etapa de diagnóstico, os 3.414 chamados com início ausente representam desistência/abandono antes do início do atendimento (`tempo_atendimento` nulo e chamado não resolvido).
- O campo foi mantido como `NULL`, refletindo que o atendimento não chegou a acontecer.




In [0]:
# Leitura da tabela bruta na camada Bronze
df_chamados_hora = spark.table(f"{bronze_schema}.tb_chamados_hora")

# Função para remover o " às " e converter para timestamp
def limpar_timestamp(coluna):
    col_limpa = F.trim(F.regexp_replace(F.col(coluna), r"[^\d/:]+", " "))
    return F.to_timestamp(col_limpa, "dd/MM/yyyy HH:mm:ss")

df_chamados_hora_silver = (
    df_chamados_hora
    .withColumnRenamed("ID_Chamado", "id_chamado")
    .withColumnRenamed("ID_Cliente", "id_cliente")
    .withColumnRenamed("Hora_Abertura_Chamado", "hora_abertura_chamado")
    .withColumnRenamed("Hora_Inicio_Atendimento", "hora_inicio_atendimento")
    .withColumnRenamed("Hora_Finalizacao_Atendimento", "hora_finalizacao_atendimento")
    .withColumn("id_chamado", F.col("id_chamado").cast("int"))
    .withColumn("id_cliente", F.col("id_cliente").cast("bigint"))
    .withColumn("hora_abertura_chamado", limpar_timestamp("hora_abertura_chamado"))
    .withColumn("hora_inicio_atendimento", limpar_timestamp("hora_inicio_atendimento"))
    .withColumn("hora_finalizacao_atendimento", limpar_timestamp("hora_finalizacao_atendimento"))
    .withColumn("_processed_at", F.current_timestamp())
    # Remoção do metadado de ingestão da camada Bronze
    .drop("_ingested_at")
)

# Gravação na Silver
(
    df_chamados_hora_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.tb_chamados_hora")
)

display(spark.table(f"{silver_schema}.tb_chamados_hora").limit(10))

## Transformações — `tb_motivos`

### 1. Nomenclatura e Tipagem
- Colunas padronizadas em *snake_case* minúsculo (`id_motivo`, `nome_motivo`, `categoria`, `criticidade`).
- `id_motivo` tipado como `int`.

### 2. Preenchimento e Padronização de Categorias
- Identificação de 3 registros com categoria ausente na base bruta, classificados conforme os critérios de negócio do case:
  - `Contratação de cartão adicional` (ID 12) → `Cartão`.
  - `Transferência de agência` (ID 14) → `Cadastral`.
  - `Cancelamento de conta` (ID 16) → `Cadastral`.

### 3. Padronização de Criticidade
- Uniformização dos valores para o padrão de capitalização com acentuação (`Alta`, `Média`, `Baixa`), eliminando as inconsistências de digitação da base bruta (`alta`, `Media`, `baixa`).


In [0]:
# Leitura da tabela bruta na camada Bronze
df_motivos = spark.table(f"{bronze_schema}.tb_motivos")

df_motivos_silver = (df_motivos
    # Tipagem da chave primária
    .withColumn("id_motivo", F.col("id_motivo").cast("int"))
    
    # Preenchimento e acentuação das 5 categorias oficiais
    .withColumn("categoria",
        F.when(F.col("id_motivo") == 12, "Cartão")
         .when(F.col("id_motivo").isin(14, 16), "Cadastral")
         .when(F.col("categoria") == "Cartao", "Cartão")
         .when(F.col("categoria") == "Beneficios", "Benefícios")
         .otherwise(F.col("categoria"))
    )
    
    .withColumn("criticidade",
        F.when(F.lower(F.col("criticidade")).isin("alta"), "Alta")
         .when(F.lower(F.col("criticidade")).isin("media", "média"), "Média")
         .when(F.lower(F.col("criticidade")).isin("baixa"), "Baixa")
         .otherwise("Não Informado")
    )
    
    .withColumn("_processed_at", F.current_timestamp())
    # Remoção do metadado de ingestão da camada Bronze
    .drop("_ingested_at")
)

# Gravação na Silver
(
    df_motivos_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.tb_motivos")
)

display(spark.table(f"{silver_schema}.tb_motivos"))

## Transformações — `tb_clientes`

### 1. Nomenclatura e Tipagem
- Renomeação da coluna `nome` para `nome_cliente`, alinhando à convenção das demais dimensões.
- `id_cliente` tipado como `bigint` para suportar o formato de CPF sem risco de overflow.
- `idade` e `dia_vencimento_fatura` tipados como `int`.

### 2. Padronização de Região
- Consolidação dos valores de `regiao` no padrão de capitalização (`Centro-Oeste`, `Nordeste`, `Norte`, `Sudeste`, `Sul`), corrigindo entradas em caixa alta da base bruta.

### 3. Tratamento de Formatos de Data
- Unificação dos formatos heterogêneos de `data_cadastro` (`dd/MM/yyyy` e `yyyy-MM-dd`) para o tipo `date`.


In [0]:
# Leitura da tabela bruta na camada Bronze
df_clientes = spark.table(f"{bronze_schema}.tb_clientes")

df_clientes_silver = (
    df_clientes
    # Padronização de nomenclatura
    .withColumnRenamed("nome", "nome_cliente")
    
    # Tipagem da chave e dos campos numéricos
    .withColumn("id_cliente", F.col("id_cliente").cast("bigint"))
    .withColumn("idade", F.col("idade").cast("int"))
    .withColumn("dia_vencimento_fatura", F.col("dia_vencimento_fatura").cast("int"))
    
    # Limpeza de texto no nome
    .withColumn("nome_cliente", F.trim(F.col("nome_cliente")))
    
    # Padronização da região
    .withColumn("regiao", F.initcap(F.col("regiao")))
    
    # Unificação dos formatos de data
    .withColumn("data_cadastro",
        F.expr("coalesce(try_to_date(data_cadastro, 'dd/MM/yyyy'), try_to_date(data_cadastro, 'yyyy-MM-dd'))")
    )
    
    .withColumn("_processed_at", F.current_timestamp())
    .drop("_ingested_at")
)

# Gravação na Silver
(
    df_clientes_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.tb_clientes")
)

display(spark.table(f"{silver_schema}.tb_clientes").limit(10))
